# SkillsAgent — Colab runner with real TSFM (T4 GPU)

This notebook runs the **SkillsAgent ablation study** on a Colab GPU so `granite-tsfm` (forecasting + anomaly detection) executes for real instead of falling back to the mock.

**Before you start**

1. `Runtime → Change runtime type → GPU (T4)`
2. Put your project folder in Google Drive at e.g. `MyDrive/HPML/project/` containing `SkillsAgent/`, `AssetOpsBench/`, and `main.json`.
3. Run cells top-to-bottom.

**What this notebook does**

- Git-clones AssetOpsBench from GitHub + rsyncs SkillsAgent & WO sample CSVs from Drive (~1 min total)
- Installs `granite-tsfm` + SkillsAgent requirements
- Extracts per-asset CSVs from `main.json` once (→ `data/chillers/`)
- Writes a Colab `.env` (no CouchDB; uses `IOT_CSV_DIR` instead)
- Smoke-tests real TSFM on the T4
- Runs `eval_runner` on the representative subset and downloads results
- Optional **Section 6b**: AssetOpsBench snapshot → ablations → external `/grade` (three cells)

In [1]:
import os, subprocess

assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, \
    "GPU not attached — Runtime → Change runtime type → T4 GPU"

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-8bbac555-da00-b8a2-5eab-d58cdae39106)



## 1. Fetch code to local disk

Drive → Colab FUSE is painfully slow for tens of thousands of small files (`.venv`, `.git` pack objects, cached artifacts), so we **git-clone AssetOpsBench from GitHub** (~30 s) and only rsync the small bits that aren't in the upstream repo:

- `AssetOpsBench/src/tmp/assetopsbench/sample_data/` — WO CSVs (~8 MB) needed for `USE_WO_SUBPROCESS`
- `SkillsAgent/` — your own code, with aggressive excludes

Total setup: ~1 minute instead of ~3 hours. If you have **local edits** to AssetOpsBench, set `ASSETOPS_REPO` below to your fork, or revert to the old rsync path.

In [2]:
import os, subprocess, time
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/HPML_Project"
LOCAL_ROOT    = "/content/work"
ASSETOPS_REPO = "https://github.com/IBM/AssetOpsBench.git"
ASSETOPS_REF  = "main"

assert os.path.isdir(DRIVE_PROJECT), f"not found: {DRIVE_PROJECT}"
os.makedirs(LOCAL_ROOT, exist_ok=True)

aob_dir = f"{LOCAL_ROOT}/AssetOpsBench"

# 1a. AssetOpsBench: shallow git clone (seconds) instead of Drive rsync (hours).
#     If a prior attempt left a non-empty dir without `.git`, git clone exits 128
#     ("destination path already exists and is not an empty directory"); we wipe
#     partials and surface stderr so any real error is actually visible.
t0 = time.time()
if os.path.isdir(aob_dir) and not os.path.isdir(f"{aob_dir}/.git"):
    print(f"  wiping partial/empty {aob_dir}")
    subprocess.check_call(["rm", "-rf", aob_dir])

if not os.path.isdir(f"{aob_dir}/.git"):
    r = subprocess.run(
        ["git", "clone", "--depth=1", "--branch", ASSETOPS_REF,
         ASSETOPS_REPO, aob_dir],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("--- git stdout ---\n" + (r.stdout or ""))
        print("--- git stderr ---\n" + (r.stderr or ""))
        raise RuntimeError(f"git clone failed (exit {r.returncode})")
    print(f"  AssetOpsBench cloned in {time.time()-t0:.1f}s")
else:
    print(f"  AssetOpsBench already at {aob_dir} — skipping clone")

# 1b. WO sample CSVs live at src/tmp/assetopsbench/sample_data/ — NOT in the
# upstream git repo, so rsync them (~8 MB) from Drive. Needed by FMSR
# (failure_codes, primary_failure_codes) and USE_WO_SUBPROCESS.
t0 = time.time()
drive_tmp = f"{DRIVE_PROJECT}/AssetOpsBench/src/tmp"
local_tmp = f"{aob_dir}/src/tmp"
if os.path.isdir(drive_tmp):
    os.makedirs(local_tmp, exist_ok=True)
    subprocess.check_call(["rsync", "-a", f"{drive_tmp}/", f"{local_tmp}/"])
    print(f"  WO sample_data rsynced in {time.time()-t0:.1f}s")
else:
    print(f"  SKIP src/tmp: not found at {drive_tmp} — WO tool will fall back to mock")

# 1c. SkillsAgent: small code-only rsync from Drive. Aggressive excludes skip
# OneDrive/MacOS junk and prior eval outputs (step 7 copies those back to Drive).
t0 = time.time()
subprocess.check_call([
    "rsync", "-a",
    "--exclude=.git", "--exclude=__pycache__", "--exclude=.pytest_cache",
    "--exclude=.venv", "--exclude=.DS_Store", "--exclude=eval_results",
    f"{DRIVE_PROJECT}/SkillsAgent/", f"{LOCAL_ROOT}/SkillsAgent/",
])
print(f"  SkillsAgent rsynced in {time.time()-t0:.1f}s")

SKILLS    = f"{LOCAL_ROOT}/SkillsAgent"
ASSETOPS  = f"{aob_dir}/src"
MAIN_JSON = f"{DRIVE_PROJECT}/main.json"
TSFM_REPORT = f"{DRIVE_PROJECT}/tsfm_report.csv"

print()
print("SKILLS   ", SKILLS)
print("ASSETOPS ", ASSETOPS)
print("TSFM_REPORT", TSFM_REPORT, "(exists)" if os.path.isfile(TSFM_REPORT) else "(MISSING)")
print("MAIN_JSON", MAIN_JSON, "(exists)" if os.path.isfile(MAIN_JSON) else "(MISSING)")

Mounted at /content/drive
  AssetOpsBench cloned in 3.1s
  WO sample_data rsynced in 69.8s
  SkillsAgent rsynced in 24.5s

SKILLS    /content/work/SkillsAgent
ASSETOPS  /content/work/AssetOpsBench/src
TSFM_REPORT /content/drive/MyDrive/HPML_Project/tsfm_report.csv (exists)
MAIN_JSON /content/drive/MyDrive/HPML_Project/main.json (exists)


## 2. Install dependencies

Two places deps need to land, because TSFM runs via `uv run` inside a separate venv:

1. **Colab system Python** — SkillsAgent's own deps (fast).
2. **`AssetOpsBench/.venv`** — `torch`, `transformers`, and `granite-tsfm` (the big one). Upstream `pyproject.toml` explicitly says *"tsfm_public must be installed separately"* and only lists `torch`/`transformers` in the optional `[tsfm]` group, so we do both.

First run: ~3–5 min (mostly pulling CUDA torch wheel). Cached on subsequent runs in the same session.

In [3]:
import subprocess, sys, time

# 2a. SkillsAgent's own Python deps → Colab system Python.
print("→ SkillsAgent requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "-r", f"{SKILLS}/requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "ijson"])

# 2b. AssetOpsBench venv gets torch + transformers + granite-tsfm so the
#     `uv run python -c ...` TSFM subprocess can import tsfm_public.
print("→ uv (drives AssetOpsBench venv)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

print("→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...")
t0 = time.time()
subprocess.check_call(["uv", "sync", "--group", "tsfm"], cwd=aob_dir)
print(f"   uv sync done in {time.time()-t0:.1f}s")

venv_py = f"{aob_dir}/.venv/bin/python"
assert os.path.isfile(venv_py), f"venv python missing: {venv_py} — did uv sync succeed?"

print("→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...")
t0 = time.time()
# --python pins the install to THIS venv; otherwise uv's resolution can land
# the package somewhere else (cache-only, or Colab system) and tsfm_public
# stays unimportable from `uv run`.
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "granite-tsfm @ git+https://github.com/ibm-granite/granite-tsfm.git",
])
print(f"   granite-tsfm installed in {time.time()-t0:.1f}s")

# transformers 4.57 pins huggingface-hub <1.0, but resolution sometimes pulls
# hf-hub 1.x (Colab system env + granite-tsfm transitive). Force back to <1.0.
print("→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...")
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "huggingface-hub<1.0",
])

# Show that the four critical packages actually landed in this venv.
r = subprocess.run(
    ["uv", "pip", "list", "--python", venv_py],
    capture_output=True, text=True,
)
hits = [ln for ln in r.stdout.splitlines()
        if ln.lower().startswith(("granite-tsfm", "torch ", "transformers ",
                                  "huggingface-hub"))]
print("venv packages:\n  " + "\n  ".join(hits) if hits else "  (none matched)")

# Final sanity check via `uv run` — mirrors how tools.py calls TSFM.
probe = (
    "import torch, tsfm_public, sys; "
    "print('torch', torch.__version__, 'CUDA?', torch.cuda.is_available(), "
    "'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', "
    "'| tsfm_public OK @', tsfm_public.__file__)"
)
r = subprocess.run(["uv", "run", "--no-sync", "python", "-c", probe],
                   cwd=aob_dir, capture_output=True, text=True, timeout=180)
print("--- venv sanity:", r.stdout.strip())
if r.returncode != 0:
    print("--- stderr:", r.stderr.strip())
    raise RuntimeError("AssetOpsBench venv missing deps")

→ SkillsAgent requirements...
→ uv (drives AssetOpsBench venv)...
→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...
   uv sync done in 16.6s
→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...
   granite-tsfm installed in 32.3s
→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...
venv packages:
  granite-tsfm                       0.3.6.dev106+gc2c7b3771
  huggingface-hub                    0.36.2
  torch                              2.10.0
  transformers                       4.57.6
--- venv sanity: torch 2.10.0+cu128 CUDA? True device: NVIDIA RTX PRO 6000 Blackwell Server Edition | tsfm_public OK @ /content/work/AssetOpsBench/.venv/lib/python3.12/site-packages/tsfm_public/__init__.py


## 3. Extract per-asset CSVs from `main.json`

One-shot: streams the 1.16 GB export and writes `chiller_6.csv` / `chiller_9.csv` (~1–2 min). `USE_IOT_SUBPROCESS` will be off, so `get_sensor_data` reads these.

In [4]:
CSV_DIR = f"{SKILLS}/data/chillers"
if not os.path.isfile(f"{CSV_DIR}/chiller_6.csv"):
    !python {SKILLS}/scripts/extract_main_json.py \
        --input {MAIN_JSON} --outdir {CSV_DIR} \
        --assets 'Chiller 6,Chiller 9' --max-rows 2000
else:
    print("CSVs already extracted — skipping")

!wc -l {CSV_DIR}/*.csv

CSVs already extracted — skipping
  1501 /content/work/SkillsAgent/data/chillers/chiller_6.csv
  1501 /content/work/SkillsAgent/data/chillers/chiller_9.csv
  3002 total


## 4. Write Colab `.env`

We skip CouchDB entirely. `IOT_CSV_DIR` drives `get_sensor_data`. `PATH_TO_MODELS_DIR` tells AssetOpsBench where the pretrained TTM checkpoints live.

**Paste your own API keys below** (watsonx is the default LLM provider).

In [5]:
from getpass import getpass

WATSONX_API_KEY    = os.environ.get("WATSONX_API_KEY")    or getpass("WATSONX_API_KEY: ")
WATSONX_PROJECT_ID = os.environ.get("WATSONX_PROJECT_ID") or input("WATSONX_PROJECT_ID: ")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")       or ""
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")     or ""

env = f"""\
LLM_PROVIDER=watsonx
WATSONX_API_KEY={WATSONX_API_KEY}
WATSONX_PROJECT_ID={WATSONX_PROJECT_ID}
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
GROQ_API_KEY={GROQ_API_KEY}
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_MODEL=gemini-2.5-flash

ASSETOPS={ASSETOPS}
PATH_TO_MODELS_DIR={ASSETOPS}/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR={CSV_DIR}
COUCHDB_EXPORT_PATH={MAIN_JSON}

USE_IOT_SUBPROCESS=0
USE_TSFM_SUBPROCESS=1
USE_WO_SUBPROCESS=1
USE_FMSR_SUBPROCESS=1
WO_CSV_DIR={ASSETOPS}/tmp/assetopsbench/sample_data
TSFM_SUBPROCESS_TIMEOUT=600
TSFM_MIN_CONTEXT_ROWS=96
"""
open(f"{SKILLS}/.env", "w").write(env)
print("wrote", f"{SKILLS}/.env")
print(env)

wrote /content/work/SkillsAgent/.env
LLM_PROVIDER=watsonx
WATSONX_API_KEY=8XywHLJHicXCTTwC1pkz7fVXrZTBKUoziGixx9Vz7ryj
WATSONX_PROJECT_ID=178e05b2-3352-4f21-8388-572e6b13d65d
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
GROQ_API_KEY=
GEMINI_API_KEY=
GEMINI_MODEL=gemini-2.5-flash

ASSETOPS=/content/work/AssetOpsBench/src
PATH_TO_MODELS_DIR=/content/work/AssetOpsBench/src/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR=/content/work/SkillsAgent/data/chillers
COUCHDB_EXPORT_PATH=/content/drive/MyDrive/HPML_Project/main.json

USE_IOT_SUBPROCESS=0
USE_TSFM_SUBPROCESS=1
USE_WO_SUBPROCESS=1
USE_FMSR_SUBPROCESS=1
WO_CSV_DIR=/content/work/AssetOpsBench/src/tmp/assetopsbench/sample_data
TSFM_SUBPROCESS_TIMEOUT=600
TSFM_MIN_CONTEXT_ROWS=96



## 5. Smoke-test real TSFM

First run compiles the model; subsequent calls are <2 s. A successful run prints `forecast source: tsfm_subprocess` and `tsad_records` > 0.

In [6]:
import sys, time
sys.path.insert(0, SKILLS)
os.chdir(SKILLS)

for m in list(sys.modules):
    if m in ("tools", "knowledge", "agent", "skills", "confidence_evaluator"):
        del sys.modules[m]

from dotenv import load_dotenv
load_dotenv(f"{SKILLS}/.env", override=True)
from tools import get_sensor_data, forecast_sensor, deep_tsfm_refine_anomalies, detect_anomaly

d = get_sensor_data("Chiller 6", lookback_days=14)
print("IoT source:", d["source"], "rows:", d.get("iot_total_observations"))

t0 = time.time()
f = forecast_sensor("Chiller 6", "flow_rate_GPM", horizon_days=7, sensor_data=d)
print(f"forecast source={f['source']}  elapsed={time.time()-t0:.1f}s  head={f.get('forecasted', [])[:3]}")

basic = detect_anomaly(d)
t0 = time.time()
deep = deep_tsfm_refine_anomalies("Chiller 6", d, basic)
print(
    f"deep tsad  refined={deep.get('deep_tsfm_refined')}"
    f"  tsad_records={deep.get('tsfm_integrated_tsad_records', 0)}"
    f"  anomalies_detected={deep.get('anomalies_detected')}"
    f"  severity={deep.get('severity')}"
    f"  n_details={len(deep.get('anomaly_details', []))}"
    f"  elapsed={time.time()-t0:.1f}s"
)

IoT source: iot_csv rows: 1500
forecast source=tsfm_subprocess  elapsed=6.0s  head=[0.691148579120636, 0.5683186054229736, 0.6224347949028015]
deep tsad  refined=True  tsad_records=45  anomalies_detected=True  severity=high  n_details=15  elapsed=6.1s


## 6a. Calibrate per-skill wall-clock latency on this GPU

Runs each skill 3× on a representative task and writes `skill_costs.json`. The evaluator picks this up automatically so Condition~E's `cost_budget` reflects T4 latencies, not Mac-CPU priors.

In [7]:
!cd {SKILLS} && python scripts/calibrate_costs.py --runs 3 --output skill_costs.json

import json
print(json.dumps(json.load(open(f'{SKILLS}/skill_costs.json')), indent=2))

Calibrating 7 skills × 3 runs (task: 'Diagnose anomalies on Chiller 6 and generate a work order')

  data_retrieval           median=  0.007s   runs=['0.634', '0.007', '0.005']
  metadata_retrieval       median=  3.092s   runs=['5.220', '2.643', '3.092']
  anomaly_detection        median=  7.103s   runs=['6.163', '7.103', '8.611']
  root_cause_analysis      median= 13.937s   runs=['13.937', '13.276', '14.137']
  validate_failure         median=  0.000s   runs=['0.000', '0.000', '0.000']
  forecasting              median= 11.943s   runs=['12.592', '10.793', '11.943']
  work_order_generation    median=  0.396s   runs=['0.383', '0.414', '0.396']

  __deep_tsfm__            median=  6.090s

wrote skill_costs.json  (36.48s total per full plan)
{
  "data_retrieval": 0.0072,
  "metadata_retrieval": 3.0918,
  "anomaly_detection": 7.1025,
  "root_cause_analysis": 13.9367,
  "validate_failure": 0.0,
  "forecasting": 11.9426,
  "work_order_generation": 0.3962,
  "__deep_tsfm__": 6.0902
}


## 6. Run eval on the representative subset

Uses the built-in `BUILTIN_TASK_BANK` (no HF download). To sweep more scenarios pass `--hf-limit 20`. All conditions A–E run; theta sweep included.

In [ ]:
OUT = f"{SKILLS}/eval_results/colab_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs(OUT, exist_ok=True)

!cd {SKILLS} && python -m eval_runner \
    --output-dir {OUT} \
    --tsfm-report {TSFM_REPORT} \
    --trajectory-log {OUT}/trajectories.jsonl 2>&1

print("\nfiles:")
!ls -la {OUT}


────────────────────────────────────────────────────────────
Condition: A_raw_llm
────────────────────────────────────────────────────────────
  TSFM_117 [fault_diagnosis] When compressor motor of Chiller 6 fails, what is the t...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=4.47s
  TSFM_120 [fault_diagnosis] I want to build an anomaly model for identifying a chil...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=5.187s
  TSFM_404 [fault_diagnosis] Get the events of equipment CWC04009 for year 2019 and ...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=4.534s
  TSFM_408 [fault_diagnosis] Get the daily count of the events of alert, anomaly for...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=3.719s
  TSFM_412 [fault_diagnosis] I would like to predict the next work order probability...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=4.668s
  TSFM_413 [fault_diagnosis] Assuming that the current date is 2018-01-02, can you p...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=1.251s
  TSFM_414 [fault

## 6b. AssetOpsBench pipeline (optional): snapshot → ablations → external grading

Same flow as `docs/assetopsbench_pipeline.md`, split into **three cells** so you can rerun stages independently:

1. **Fetch** scenarios from the scenario server into a JSONL snapshot (`scripts/export_scenarios.py`).
2. **Run** SkillsAgent ablations on that file (`eval_runner.py --scenario-file …`), producing `ablation_results.csv` with `result_json` / `trace_json`.
3. **Grade** via `scripts/score_with_assetopsbench.py --from-eval-csv` (POST to `/scenario-set/{id}/grade`).

**Connectivity:** Colab usually cannot reach `http://localhost:8099` on your laptop. Set **`AOB_SCENARIO_SERVER`** to a URL this runtime can reach (ngrok, SSH tunnel, or run `serve.py` where it is routable). Example: `os.environ["AOB_SCENARIO_SERVER"] = "https://abc123.ngrok.io"` before cell 1.

Optional env overrides: **`AOB_SCENARIO_LIMIT`** (integer max scenarios), **`AOB_CONDITIONS`** (space-separated, default `C D`).

In [ ]:
# --- 6b-1) Fetch scenarios from AssetOpsBench scenario server ---
import os
import subprocess
import sys

AOB_SCENARIO_SERVER = os.environ.get(
    "AOB_SCENARIO_SERVER",
    "http://localhost:8099",
).rstrip("/")

TSFM_SCENARIO_SET = os.environ.get(
    "AOB_SCENARIO_SET",
    "13aab653-66fe-4fe6-84d8-89f1b18eede3",
)

SNAPSHOT_PATH = os.path.join(SKILLS, "eval_inputs/colab_aob_snapshot/scenarios.jsonl")

_export_cmd = [
    sys.executable,
    os.path.join(SKILLS, "scripts/export_scenarios.py"),
    "--server-url",
    AOB_SCENARIO_SERVER,
    "--scenario-set",
    TSFM_SCENARIO_SET,
    "--output",
    SNAPSHOT_PATH,
]
_lim = os.environ.get("AOB_SCENARIO_LIMIT", "").strip()
if _lim.isdigit():
    _export_cmd.extend(["--limit", _lim])

print("Server:", AOB_SCENARIO_SERVER)
print("Fetch →", SNAPSHOT_PATH)
subprocess.check_call(_export_cmd)

In [ ]:
# --- 6b-2) Run SkillsAgent ablations on the snapshot ---
import os
import subprocess
import sys
import time

PIPE_RUN_OUT = os.path.join(
    SKILLS,
    "eval_results",
    f"colab_aob_pipe_{time.strftime('%Y%m%d_%H%M')}",
)
CONDITIONS = (
    os.environ.get("AOB_CONDITIONS", "C D").strip().split()
)

_eval_cmd = [
    sys.executable,
    "-m",
    "eval_runner",
    "--output-dir",
    PIPE_RUN_OUT,
    "--scenario-file",
    SNAPSHOT_PATH,
    "--scenario-set-id",
    TSFM_SCENARIO_SET,
    "--conditions",
    *CONDITIONS,
    "--trajectory-log",
    os.path.join(PIPE_RUN_OUT, "trajectories.jsonl"),
]
print("Output:", PIPE_RUN_OUT)
print("Conditions:", CONDITIONS)
subprocess.check_call(_eval_cmd, cwd=SKILLS)
print("CSV:", os.path.join(PIPE_RUN_OUT, "ablation_results.csv"))

In [ ]:
# --- 6b-3) Submit ablation CSV to AssetOpsBench grader ---
import os
import subprocess
import sys

GRADE_OUT = os.path.join(PIPE_RUN_OUT, "assetopsbench_graded")
_csv = os.path.join(PIPE_RUN_OUT, "ablation_results.csv")

_grade_cmd = [
    sys.executable,
    os.path.join(SKILLS, "scripts/score_with_assetopsbench.py"),
    "--from-eval-csv",
    _csv,
    "--scenario-set",
    TSFM_SCENARIO_SET,
    "--server-url",
    AOB_SCENARIO_SERVER,
    "--conditions",
    *CONDITIONS,
    "--output-dir",
    GRADE_OUT,
]
subprocess.check_call(_grade_cmd, cwd=SKILLS)
print("Graded summary/details:", GRADE_OUT)

## 7. Copy results back to Drive

So you keep them after the runtime is recycled.

In [ ]:
DRIVE_OUT = f"{DRIVE_PROJECT}/SkillsAgent/eval_results/{os.path.basename(OUT)}"
os.makedirs(DRIVE_OUT, exist_ok=True)
subprocess.check_call(["rsync", "-a", f"{OUT}/", f"{DRIVE_OUT}/"])
print("copied →", DRIVE_OUT)